In [2]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
from torchvision.ops import box_area
from collections import Counter

In [9]:
#lab_df = pd.read_csv('wf_labelled_data.csv')
lab_df = pd.read_csv('../Downloads/data-2022-08-01.csv')

In [10]:
labeler_root = 'GitHub/neal/www'
labeler_list = [l for l in os.listdir(labeler_root) if l not in ['tmp', 'JS'] and "." not in l]
labeler_list

['Anthony',
 'BatProject',
 'donohui',
 'kinge6',
 'markshorten',
 'nialltkeogh',
 'omarcaif']

In [11]:
lab_prefix = [v.split('@')[0] for v in lab_df.labeler]
lab_df['lab_folder'] = [l if l in labeler_list else 'tmp' for l in lab_prefix]
lab_df = lab_df[lab_df.labeler.str.startswith("markshorten")]

In [12]:
audio_paths = [os.path.join('wf_data_clips', r.file_name) for i,r in lab_df.iterrows()]
audio_paths_exists = [os.path.exists(p) for p in audio_paths]
#do they all exist
np.mean(audio_paths_exists)

1.0

In [13]:
pop_classes = ["Blackbird",
"Chaffinch", 
"Dunnock",
"Goldfinch",
"Hooded Crow",
"House Sparrow", 
"Linnet",
"Meadow Pipit",
"Pied Wagtail", 
"Robin", 
"Rook",
"Sedge Warbler", 
"Starling",
"Stonechat",
"Wren",
"Yellowhammer"]

In [14]:
df = lab_df[audio_paths_exists].copy()
df.class_label = df.class_label.str.capitalize()
df_popclasses = df[df.class_label.isin(pop_classes)]

In [15]:
def bbox_fn(x):
    bboxes = torch.tensor(x[['start_time', 'start_freq', 'end_time', 'end_freq']].values)
    return box_area(bboxes).sum().item()

In [16]:
df_bboxes = df_popclasses.groupby(["file_name", "class_label"]).apply(bbox_fn).reset_index()
df_bboxes.columns = df_bboxes.columns[:-1].to_list() + ["bbox_sum"]

df_bboxes=df_bboxes.sort_values(by=['file_name', 'bbox_sum'])#, drop_duplicates()
df_singlelabs = df_bboxes.drop_duplicates("file_name", keep = "last")

In [17]:
df_singlelabs

,file_name,class_label,bbox_sum
1,CARNSOREMET_20220427_154300_start_2_34.wav,Stonechat,11.350224
2,CARNSOREMET_20220427_160300_start_0_28.wav,Stonechat,29.043712
3,CARNSOREMET_20220427_160300_start_1_24.wav,Blackbird,1.937444
4,CARNSOREMET_20220427_160300_start_3_2.wav,Goldfinch,35.101278
6,CARNSOREMET_20220427_161300_start_3_2.wav,Goldfinch,18.235895
...,...,...,...
500,TEEVURCHER_20220530_183500_start_3_30.wav,Robin,19.667733
501,TEEVURCHER_20220530_183500_start_3_44.wav,Blackbird,19.715765
502,TEEVURCHER_20220530_183500_start_3_58.wav,Robin,12.470815
503,TEEVURCHER_20220530_190500_start_1_52.wav,Blackbird,10.344314


In [18]:
counter = Counter(df_singlelabs.class_label)
print(counter)

Counter({'Blackbird': 136, 'Wren': 69, 'Robin': 46, 'Linnet': 37, 'Chaffinch': 34, 'Dunnock': 20, 'Yellowhammer': 13, 'Rook': 13, 'Starling': 11, 'Stonechat': 9, 'Goldfinch': 6})


In [19]:
counter['Blackbird']

136

In [23]:
#class_counts = {0: 133, 14: 74, 9: 41, 6: 38, 1: 28, 2: 17, 10: 16, 12: 11, 15: 11, 13: 9, 3: 5}
max_classes = 200

In [32]:
class_missing = {p: max_classes - counter[p] for p in counter.keys()}
class_missing

{'Stonechat': 191,
 'Blackbird': 64,
 'Goldfinch': 194,
 'Linnet': 163,
 'Dunnock': 180,
 'Wren': 131,
 'Chaffinch': 166,
 'Robin': 154,
 'Yellowhammer': 187,
 'Rook': 187,
 'Starling': 189}

In [35]:
import json
import requests
from pydub import AudioSegment

In [113]:
def xeno_download(search_string, abbrev_string, top_n = 250, ssp = None, not_ssp = None, start_date = None):
    folder_sv = 'xenobirds/'+search_string
    find_bracket = (search_string.find(' ('))
    if (ssp is not None) & (find_bracket == -1):
        folder_sv = folder_sv + ' ('+ssp.capitalize()+')'
    folder_sv = folder_sv +'/'
    if not os.path.isdir(folder_sv):
        os.mkdir(folder_sv)
    if abbrev_string is None:
        if ssp is not None:
            abbrev_string = (search_string+ssp).replace(' ', '_').lower()
        else:
            abbrev_string = search_string.replace(' ', '_').lower()
    if find_bracket != -1:
        search_string = search_string[:find_bracket]
    search_str = search_string.replace(' ', '%20').lower()
    if ssp is not None:
        url = 'https://www.xeno-canto.org/api/2/recordings?query=ssp:'+ssp
    else:
        url = 'https://www.xeno-canto.org/api/2/recordings?query='+search_str
    doc = requests.get(url)
    #with open(abbrev_string+'-query.json', 'wb') as f:
    #    f.write(doc.content)
    
    # Get the json entries from your downloaded json
    #jsonFile = open(abbrev_string+'-query.json', 'r')
    #values = json.load(jsonFile)
    #jsonFile.close()
    values = json.loads(doc.content.decode())
    # Create a pandas dataframe of records & convert to .csv file
    record_df = pd.DataFrame(values['recordings'])
    print(len(record_df))
    in_country = record_df.cnt.isin(["Ireland", "United Kingdom"])
    record_df = pd.concat([record_df[in_country], record_df[~in_country]])
    print(len(record_df))
    #record_df.to_csv('xc-'+abbrev_string+'.csv', index=False)
    #print(record_df)
    #return None
    #record_df = record_df[record_df.en.str.startswith(search_string)]
    #record_df.loc[np.logical_or(record_df.en.str.startswith(search_string), 
    #                np.logical_and(record_df.en.str.endswith(search_string), 
    #                    record_df.en.str.startswith(("European", "Common"))))]
    if start_date is not None:
        record_df = record_df[record_df.uploaded >= start_date]
    if not_ssp is not None:
        record_df = record_df[record_df.ssp != not_ssp]
    #return record_df
    # Make wget input file
    url_list   = []
    fname_list = []
    for file, fname in zip(record_df['file'].tolist(), record_df['file-name'].tolist()):
        if file.startswith('http'):
            url_list.append(file)
        else:
            url_list.append('https:{}'.format(file))
        fname_list.append(fname.split('.')[-1])
    with open('xc-'+abbrev_string+'-urls.txt', 'w+') as f:
        for item in url_list:
            f.write("{}\n".format(item))
    #print(record_df[record_df.file == url_list[0]])
    #print(record_df['file-name'])
    #print(record_df.columns)
    #return None        
    txt_file = open('xc-'+abbrev_string+'-urls.txt', 'r')
    url_list = [s.replace('\n', '') for s in txt_file.readlines()]
    #
    url_list, fname_list = url_list[:top_n], fname_list[:top_n]
    # 
    print(len(url_list))
    #return None
    for url_i, fname_i in zip(url_list, fname_list):
        filename_sv = folder_sv+'xc'+url_i.split('/')[3]+'.mp3'
        filename_sv_wv = filename_sv[:-3]+'wav'
        #if filename_sv_wv in 
        if filename_sv in os.listdir(folder_sv):
            continue
        if filename_sv_wv in os.listdir(folder_sv):
            continue
        doc = requests.get(url_i)
        with open(filename_sv, 'wb') as f:
            f.write(doc.content)
        if fname_i in ["mp3", "MP3"]:
            sound = AudioSegment.from_mp3(filename_sv)
            sound.export(filename_sv_wv, format="wav")
            os.remove(filename_sv)

In [114]:
for c,n_miss in class_missing.items():
    print(c, n_miss)
    abbrev_string = c.lower()[:2]
    xeno_download(c, abbrev_string, top_n = n_miss)
    #break

Stonechat 191
500
500
191
Blackbird 64
500
500
64
Goldfinch 194
500
500
194
Linnet 163
500
500
163
Dunnock 180
500
500
180
Wren 131
500
500
131
Chaffinch 166
500
500
166
Robin 154
500
500
154
Yellowhammer 187
500
500
187
Rook 187
500
500
187
Starling 189
500
500
189


In [104]:
'https://xeno-canto.org/346446/download'.split('/')[-2]

'346446'

In [42]:
txt_file = open('xc-'+abbrev_string+'-urls.txt', 'r')
url_list = [s.replace('\n', '') for s in txt_file.readlines()]
url_list

['https:https://xeno-canto.org/346446/download',
 'https:https://xeno-canto.org/346441/download',
 'https:https://xeno-canto.org/153273/download',
 'https:https://xeno-canto.org/334286/download',
 'https:https://xeno-canto.org/30197/download',
 'https:https://xeno-canto.org/30185/download',
 'https:https://xeno-canto.org/30163/download',
 'https:https://xeno-canto.org/18725/download',
 'https:https://xeno-canto.org/760492/download',
 'https:https://xeno-canto.org/760491/download',
 'https:https://xeno-canto.org/760041/download',
 'https:https://xeno-canto.org/760040/download',
 'https:https://xeno-canto.org/760039/download',
 'https:https://xeno-canto.org/760038/download',
 'https:https://xeno-canto.org/760037/download',
 'https:https://xeno-canto.org/760036/download',
 'https:https://xeno-canto.org/760034/download',
 'https:https://xeno-canto.org/756406/download',
 'https:https://xeno-canto.org/756398/download',
 'https:https://xeno-canto.org/756397/download',
 'https:https://xeno-can